# PDF Data Curation for DAPT (Domain Adaptive Pre-Training)

#### By Janaki Vamaraju, Sugandha sharma, Aastha Jhunjhunwala, May 2024. 
#### Contact jvamaraju@nvidia.com or sugandhas@nvidia.com for questions.

This notebook walks through the processing pipeline for pdf data curation required for DAPT. Given a directory containing a set of folders and files, this pipeline performs pdf data collection and cleaning. In this notebook, we will use the datasets in the `dapt-data-curation/datasets/pdfs/` folder to illustrate pdf data curation through this pipeline.

The notebook follows the steps below:<br>
- Step 1: Install requirements, clean pdf files, convert to txt and compress<br>
- Step 2: Find all pdf text files in the given directory tree <br>
- Step 3: Examine the identified files<br>
- Step 4: Filter files by file type<br>
- Step 5: Copy all files we want to include and compress them<br>
- Step 6: Check for duplicate files<br>
- Step 7: Remove duplicate files<br>

#### The complete pipeline along with pdf processing is illustrated in the schematic shown below:

![pipeline](imgs/pipeline_pdf.png)



## Step 1: Install requirements

In [ ]:
# install all required packages for data curation
! pip3 install -r ../requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nvidia-ammo 0.0.0 requires onnx-graphsurgeon, which is not installed.
nvidia-ammo 0.0.0 requires onnxruntime~=1.16.3, which is not installed.
albumentations 1.4.4 requires numpy>=1.24.4, but you have numpy 1.24.3 which is incompatible.
albumentations 1.4.4 requires typing-extensions>=4.9.0, but you have typing-extensions 4.5.0 which is incompatible.
bokeh 3.4.1 requires contourpy>=1.2, but you have contourpy 1.0.7 which is incompatible.
cudf 23.12.0 requires pandas<1.6.0dev0,>=1.3, but you have pandas 2.0.1 which is incompatible.
cudf-cu12 23.10.2 requires pandas<1.6.0dev0,>=1.3, but you have pandas 2.0.1 which is incompatible.
cugraph 23.12.0 requires dask-cuda==23.12.*, but you have dask-cuda 23.10.0 which is incompatible.
cugraph-service-server 23.12.0 requires dask-cuda==23.12.*, but you have dask-cuda 23.10.0

In [21]:
# install any libraries needed 
! pip install pandas
! pip install matplotlib
! pip install cchardet

  Using cached cchardet-2.1.7-cp38-cp38-manylinux2010_x86_64.whl.metadata (7.7 kB)
Using cached cchardet-2.1.7-cp38-cp38-manylinux2010_x86_64.whl (265 kB)


Before getting started navigate to `dapt-data-curation/chipgpt/pdfcleaner` and build a docker containe with `./docker/`. After building the container install pdf cleaner using `pip install .` Run the docker container `docker run -it -p 8080:8080 --rm --gpus all --ipc=host --network host -v $(pwd):/workspace pdf_cleaner:latest` to run this notebook. You are now ready to process open-source research papers for this playbook. 

In [1]:
# Provide path to input folder containing pdfs 
ACADEMIC_IN="/workspace/datasets/pdfs/"
# Provide path to processed pdfs. Text from pdfs are extracted, filtered, and stored in tar files. 
ACADEMIC_OUT="/workspace/datasets/pdfs_txt/"

In [10]:
# Text from pdfs are extracted, filtered, and stored in tar files.
! python3 /workspace/chipgpt/pdfcleaner/convert.py --path $ACADEMIC_IN --outpath $ACADEMIC_OUT --filter academic

Output file /workspace/datasets/pdfs_txt/image_retrieval.pdf.txt.gz already exists. Skipping...
Output file /workspace/datasets/pdfs_txt/Deep_Learning_Toolkit-Accelerated_Analytical_Co-optimization_of_CNN_Hardware_and_Dataflow.pdf.txt.gz already exists. Skipping...
Output file /workspace/datasets/pdfs_txt/K-Means_Hashing_An_Affinity-Preserving_Quantization_Method_for_Learning_Binary_Compact_Codes.pdf.txt.gz already exists. Skipping...
Output file /workspace/datasets/pdfs_txt/DANCE_Differentiable_Accelerator_Network_Co-Exploration.pdf.txt.gz already exists. Skipping...
Output file /workspace/datasets/pdfs_txt/A_bundle_approach_to_efficient_MAP-inference_by_Lagrangian_relaxation.pdf.txt.gz already exists. Skipping...
Output file /workspace/datasets/pdfs_txt/ViraEye_Energy-Efficient_Stereo_Vision_Accelerator_with_Binary_Neural_Network_in_55_nm_CMOS.pdf.txt.gz already exists. Skipping...
Output file /workspace/datasets/pdfs_txt/decc.pdf.txt.gz already exists. Skipping...
Output file /works

In [11]:
# Delete all fins with fewer than 25 non-numerical words
# Only check fins with fewer than 250 total words
! bash rm_pdfs.sh

## Step 2: Find all pdf converted text files in the given directory tree

Now we will run the `create_manifest_non_tree.sh` script that takes in a directory as input, finds all the text files in the directory tree, and produces a manifest csv file containing a list of file names (file paths), file sizes, and number of lines in each of the text files found in the directory tree. 
- Input1: `directory`: the path to the input collection directory that has the data we want to search through
- Input2: `manifest_file`: the path to the output csv file
- output: `manifest.csv`: a csv file with three columns: Path (listing file names), Size (listing file sizes) and Lines (listing number of lines in each file) is saved to the  `dapt-data-curation/datasets/output` folder

In [2]:
# specify path to the input directory that has the data that we want to search through
directory = "../../../datasets/pdfs_txt/"

# create a folder for storing output files under the dapt-data-curation directory
! mkdir ../../../output

# specify the path to (and the name of) the output csv file
manifest_file  = "../../../output/manifest_pdf.csv"

mkdir: cannot create directory ‘../../../output’: File exists


In [3]:
# create the manifest csv file with a list of all text files found. This csv will have three columns listing file names, their sizes and number of lines in each of them
! bash ../create_manifest_non_tree.sh $directory $manifest_file

Output file already exists: ../../../output/manifest_pdf.csv


#### This is what the `dapt-data-curation/output` folder should look like at the end of Step 2:

![pipeline](imgs/output_step2.png)

In [4]:
# use pandas to display content 
import pandas
df = pandas.read_csv(manifest_file)
with pandas.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):  # more options can be specified also
    print(df)

                                                                                                                                                           Path   
0   /home/jvamaraju/dapt-data-curation/datasets/pdfs_txt/ViraEye_Energy-Efficient_Stereo_Vision_Accelerator_with_Binary_Neural_Network_in_55_nm_CMOS.pdf.txt.gz  \
1     /home/jvamaraju/dapt-data-curation/datasets/pdfs_txt/Deep_Learning_Toolkit-Accelerated_Analytical_Co-optimization_of_CNN_Hardware_and_Dataflow.pdf.txt.gz   
2                         /home/jvamaraju/dapt-data-curation/datasets/pdfs_txt/A_bundle_approach_to_efficient_MAP-inference_by_Lagrangian_relaxation.pdf.txt.gz   
3              /home/jvamaraju/dapt-data-curation/datasets/pdfs_txt/AutoFlex_Unified_Evaluation_and_Design_Framework_for_Flexible_Hybrid_Electronics.pdf.txt.gz   
4                                                                                          /home/jvamaraju/dapt-data-curation/datasets/pdfs_txt/decc.pdf.txt.gz   
5                     

## Step3 (optional): Examine the identified files

Here, we will run `report_manifest.py` that enables us to examine the largest files from the indetified files listed in the manifest csv. It takes the manifest csv as input and reports the top 10 biggest files along with their corresponding percentile sizes. It allows visualization through text based histograms for various file types. 

- Input: `manifest_pdf.csv`: a csv file with three columns: Path (listing file names), Size (listing file sizes) and Lines (listing number of lines in each file)
- Output:<br> 
       - Top 10 biggest files along with their corresponding percentile sizes <br>
       - List of the top 10 biggest files with their paths, number of lines and file size<br>
       - List of files types greater than 5MB in size and number of such files for each file type <br>
       - Histograms for various file types showing the distribution of file sizes
- Output: `file_size_histogram.png`: saves the file size histogram in the `dapt-data-curation/output` folder


In [12]:
# run the report_manifest.py script, with the manifest csv file as an argument passed to the script
! python3 ../report_manifest.py $manifest_file

10 Files
Top 10 percentiles (Size_MB):
0.90    0.059761
0.91    0.061157
0.92    0.062553
0.93    0.063950
0.94    0.065346
0.95    0.066743
0.96    0.068139
0.97    0.069536
0.98    0.070932
0.99    0.072329
Name: Size_MB, dtype: float64
Top 10 files (size):
/home/jvamaraju/dapt-data-curation/datasets/pdfs_txt/ViraEye_Energy-Efficient_Stereo_Vision_Accelerator_with_Binary_Neural_Network_in_55_nm_CMOS.pdf.txt.gz : 184 Lines 0.004962MB
/home/jvamaraju/dapt-data-curation/datasets/pdfs_txt/decc.pdf.txt.gz : 451 Lines 0.021462MB
/home/jvamaraju/dapt-data-curation/datasets/pdfs_txt/AutoFlex_Unified_Evaluation_and_Design_Framework_for_Flexible_Hybrid_Electronics.pdf.txt.gz : 578 Lines 0.022931MB
/home/jvamaraju/dapt-data-curation/datasets/pdfs_txt/K-Means_Hashing_An_Affinity-Preserving_Quantization_Method_for_Learning_Binary_Compact_Codes.pdf.txt.gz : 686 Lines 0.0243MB
/home/jvamaraju/dapt-data-curation/datasets/pdfs_txt/image_retrieval.pdf.txt.gz : 666 Lines 0.028986MB
/home/jvamaraju/dapt

### This is what the `dapt-data-curation/output` folder should look like at the end of Step 3:

![output_3](imgs/output_step3.png)

## Step4: Filter files by file type

Here we will run `process_manifest.py` to filter the files listed in the manifest csv. This will categorize the files based on their file types and generate separate csv files for each of the categories.

- Input: `manifest.csv`: a csv file with three columns: Path (listing file names), Size (listing file sizes) and Lines (listing number of lines in each file)
- Output: multiple filtered manifest csv files with one file for each of the following categories e.g., `manifest_filtered_cpp.csv`. The filtered csv files are saved in the `dapt-data-curation/output` folder

Following the are categories that the files are split into based on the file extensions: <br>

| **Category** | **File Extensions** |
|-------------------|---------------------|
| Viva              | ['.VX', '.VXH']     |
| VerilogVHDL       | ['.V', '.VH', '.VHDL'] |
| CPP               | ['.C', '.CPP', '.H', '.HPP'] |
| Python            | ['.PY']             |
| SV                | ['.SV']             |
| GV                | ['.GV']             |
| Config            | ['.CONFIG']         |
| Makefile          | ['Makefile', 'Makeppfile', '.mk'] |
| Perl              | ['.PM', '.PL']      |
| Tcl               | ['.TCL']            |
| Spec              | ['.spec']           |
| Yaml              | ['.yaml', '.yml']   |
| Spice             | ['.sp', '.cir', '.cmd', '.spf'] |
| VerilogAnalog     | ['.va']             |
| Skill             | ['.il']             |
| PT                | [".ptsh", ".proc"]  |
| EP3               | [".ep3", ".ep3.pp"] |
| Script            | ['NO_EXTENSION']    |



In [5]:
# run the process_manifest.py script, with the manifest csv file as an argument passed to the script
! python3 ../process_manifest.py $manifest_file


# Text Summary (['.TXT'])
	Total number of files: 10
	Total Size: 0 MB (uncompressed)


Total Size (uncompressed): 0 MB


#### This is what the `dapt-data-curation/output` folder should look like at the end of Step 4:

![output_4](imgs/output_step4.png)

## Step 5: Copy all files we want to include and compress them

Here we will run `execute_manifest.py` which takes a manifest csv as input and copies over all the files listed in the csv, gzips them and calculates the md5sum. We can either use the original manifest csv if we want to include all the text files we found in the directory tree for DAPT training. 

Alternatively, we can also use any of the filtered manifest csv files generated in the previous step in we want to selectivley use a particular category of files for DAPT training. 

In this example, we just use the original manifest csv to illustrate processing of a larger dataset. 

- Input: `manifest.csv`: a csv file with three columns: Path (listing file names), Size (listing file sizes) and Lines (listing number of lines in each file)
- Output: manifest folder containing each file listed in the input csv gzipped (.gz). In addition this folder contains a .txt file listed the origianl paths from where each of these files was copied. The manifest folder is saved in the `dapt-data-curation/output` folder.

In [7]:
# run the process_manifest.py script, with the manifest csv file as an argument passed to the script
! python3 ../execute_manifest.py $manifest_file

Done.


#### This is what the `dapt-data-curation/output` folder should look like at the end of Step 4:

![output_5](imgs/output_step5.png)

#### This is what the `dapt-data-curation/output/manifest` folder should look like at the end of Step 4:

![output_5_manifest](imgs/manifest_folder.png)

#### This is what the `dapt-data-curation/output/manifest/info.txt` file should look like at the end of Step 4:

![output_5_manifest_info](imgs/manifest_folder_txt.png)

## Step 6: Check for duplicate files 

Here we run `report_exact_dedup.py` in order to check for duplicate files within a specific category of files. 

- For illustration, we will first run `execute_manifest.py` on a particular category of files by using the corresponding filtered manifest file generated in Step 4. This will create a `manifest_filtered_<category_name>` folder in `dapt-data-curation/output` folder whoose contents should be similar to the output of Step 5 shown above. 


- We will then take the `info.txt` file from `manifest_filtered_<category_name>` folder and provide it as input to `report_exact_dedup.py` to check whether there are any duplicte files within this category.

- Any duplicate files found will be printed out with a list of all duplicate file paths. If the script executes silently, this implies that there are no duplicate files present.

### Example 1

In [1]:
# Run execute_manifest.py on a manifest_filtered_cpp.csv generated in Step 4 to generate the filtered_manifest folder with gzipped files of type ['.C', '.CPP', '.H', '.HPP']
# This will output a manifest_filtered_cpp folder in dapt-data-curation/output directory
filtered_manifest = "../../../output/manifest_filtered_cpp.csv"
! python3 ../execute_manifest.py $filtered_manifest

Done.


In [4]:
# Now we will run report_exact_dedup.py to check for duplicates of files of type ['.C', '.CPP', '.H', '.HPP']
# All duplicates will be printed. Empty output implies there are no duplicate files.
info_file = "../../../output/manifest_filtered_cpp/info.txt"
! python3 ../report_exact_dedup.py $info_file

### Example 2

In [5]:
# Run execute_manifest.py on a manifest_filtered_verilogvhdl.csv generated in Step 4 to generate the filtered_manifest folder with gzipped files of type ['.V', '.VH', '.VHDL']
# This will output a manifest_filtered_verilogvhdl folder in dapt-data-curation/output directory
filtered_manifest = "../../../output/manifest_filtered_verilogvhdl.csv"
! python3 ../execute_manifest.py $filtered_manifest

Done.


### Example 3

In [8]:
# Run execute_manifest.py on a manifest_filtered_pdf.csv generated in Step 4 to generate the filtered_manifest folder with gzipped files of type ['.txt', '.pdf']
# This will output a manifest_filtered_pdf folder in dapt-data-curation/output directory
filtered_manifest = "../../../output/manifest_pdf_filtered_text.csv"
! python3 ../execute_manifest.py $filtered_manifest

Done.


In [9]:
# Now we will run report_exact_dedup.py to check for duplicates of files of type ['.V', '.VH', '.VHDL']
# All duplicates will be printed. Empty output implies there are no duplicate files.
info_file = "../../../output/manifest_pdf_filtered_text/info.txt"
! python3 ../report_exact_dedup.py $info_file